# King Domino Projekt - Overblik

### Arkitekturoversigt
Projektets formål er at digitalisere og point-score fysiske King Domino-spilleplader. Projektet består af fem hovedfaser:
1. **Gitteropdeling (Grid Splitting):** 
    - Opdeling af hele pladebilleder i et 5x5 logisk gitter af terræn-tiles.
2. **Feature-ekstraktion:** 
    - Behandling af tærren-tiles for at udtrække features til en 20-værdi vektor (HSV-histogram logik)
    - Gemme 20-værdi vektorerne i en csv-fil til modeltræning/analyse.
3. **Tile-multiklassificering:** 
    - Identificering af tile-terræntypen (Water, Field, Grass, Forest, Mine, Swamp, Home, Empty). 
    - Altså en multi-class terrænklassificering udført med enten SVM eller kNN
4. **Krone-detektion:** 
    - Tælle antallet af score-multiplikatorer (kroner) til stede på hver klassificeret terræn-tile ved at bruge template matching (fx med `high_res_crown.png` / `low_res_crown.png`). 
    - Behandle gråtone/blobs
    - Håndtering af gennemsigtige "Don't care"-pixels.
5. **Spillogik og Scoreberegning:** 
    - Evaluering af 2D-arrays, der repræsenterer spillepladernes 5x5-gitter.
    - Hver spilleplade (fx med Breadth-First Search algoritme) til at beregne den endelige score for regioner (forbundne terræn-tiles).
    - Baseret på Kingdomino-reglerne(fx 6 forbundne vand-tiles * 2 kroner = 12 point).
6. **Ground Truths(GT):**
    - GT_BoardPointScore (Done!)
    - GT_BoardTileMatrix (Done!)
    - GT_CrownsPerBoard (Done!)
    - GT_CrownsPerTile



### Nuværende status

| Modul | Status | Beskrivelse |
| :--- | :---: | :--- |
| `board_split.py` | 🟢 | Deler spilleplader op i et 5x5 tile-gitter. |
| `feature_extraction.py` | 🟢 | Udtrækker 20-værdi vektor HSV-histogrammer fra tiles og gemmer data i et dataset (`features.csv`). |
| `extract_tile_feature_vector.py` | 🟡 | Indlæser billeder, opdeler tiles og udfører feature-ekstraktion |
| `classify_tile.py` | 🟢 | Implementerer `TileClassifier` ved hjælp af en Scikit-learn k-Nearest Neighbors (kNN) model trænet på `features.csv`. |
| `detect_crown.py` | 🟡 | (`CrownDetector`). Kræver kontur/form-detektion for at tælle 0-3 kongekroner per tile. |
| `scoring.py` | 🔴 | Stub (`ScoreCalculator`). Kræver BFS-algoritmen til at gange størrelsen af forbundne regioner med antal kroner. |

### Manglende dele
1. **Krone-tællelogik:** Identificering af specifikke gule konturer/former på ikke-tomme tiles for at fungere som multiplikatorer (`detect_crown.py`).
2. **Spilleregel-motor:** Implementering af BFS-gittergennemløb for at gruppere tilstødende matchende terræner og beregne den endelige score (`scoring.py`).
3. **End-to-End Integration:** Modificering af `extract_tile_feature_vector.py` til at bruge klassificeringsmodellerne og sende det resulterende 5x5 gitter af objekter til scoringsmotoren.

### Anbefalet næste skridt
**Implementer krone-detektion (`detect_crown.py`)**
Nu hvor vi kan ekstrahere features og klassificere terræntypen for hver tile med en ML-model, er den næste komponent at bygge krone-detektionen. Dette vil identificere score-multiplikatorer.

**Næste kode der skal genereres:**
Vi bør udfylde logikken indeni `CrownDetector` (`detect_crown.py`) for at detektere og tælle forekomster af gyldne kroner via template matching eller konturdetektion, med højde for potentiel støj / overlappende elementer.

## 1. Dataforbehandling og split-strategi
For at undgå datalækage laves split konsekvent på spilleplade-niveau og ikke på tile-niveau. Parrede billeder (med og uden 3D-objekter) placeres altid i samme split.

* **Samlet antal spilleplader:** 36 unikke spilleplader.
* **Ugyldige data(støj):** Billede 71, 73 og 74 er udeladt, fordi de er dubletter af billede 68, 69 og 70.
* **Dataekstraktion:** ca. 1775 tiles (71 gyldige billeder × 25 felter).

### Endelig split:
* **Hold-out testsæt:** 6 spilleplade-grupper (12 billeder i alt) reserveres til den endelige evaluering for at teste præstation af ML-model.

Test-sæt:
Spilleplader
  - (1,5), (19,23), (25,29), (35,39), (49,53), (67,70)

* **Træning og validering (grouped CV):** De resterende 30 spilleplade-grupper bruges til 5-fold grouped cross-validation.
  * Hver fold indeholder 6 spilleplade-grupper.
  * I hver CV-runde bruges 4 folds til træning og 1 fold til validering.

Fold 1:
Spilleplader 
  - Mixed_data: (4,8), (20,24), (34,38), (42,46), (48,52), (65,72)
  - Uden3dObjekt(clean_data): 3, 20, 34, 42, 48, 65
  - Med3dObjekt(dirty_data): 8, 24, 38, 46, 52, 72

Fold 2:
Spilleplader
(2,6), (18,22), (28,32), (36,40), (51,55), (58,62)

Fold 3:
Spilleplader
(10,14), (11,15), (26,30), (41,44), (57,61), (64,68)

Fold 4:
Spilleplader
(3,7), (17,21), (27,31), (43,47), (50,54), (59,63)

Fold 5:
Spilleplader
(9,13), (12,16), (33,37), (45), (56,60), (66,69)


## 2. Arkitektonisk Gennemgang og Næste Skridt (Opdateret)

Efter en gennemgang af kodens nuværende tilstand, er her mine observationer som ML-ingeniør:

### Styrker i nuværende setup:
* **Modulær opbygning:** Koden er logisk opdelt i `board_split.py`, `feature_extraction.py`, `detect_crown.py`, `classify_tile.py` og `scoring.py`. Det gør det nemt at teste komponenterne isoleret.
* **Evaluering og Data Split:** Der er en meget fornuftig tilgang til evaluering (Grouped CV på board-niveau). Dette sikrer, at vi undgår datalækage ("data leakage"), hvilket er et klassisk problem i billedanalyse.

### Fundne mangler & fejl:
1. **Fejl i `scoring.py` (Kritisk):** 
   - BFS-algoritmen i `explore_region` returnerer resultatet *inde* i `while`-løkken, hvilket gør at den kun kigger på første tile i en region før den afbrydes.
   - Variabel-navne i `parse_tile_position` matcher ikke argumenterne (`tile_name` vs `tile_filename`). 
   - Den generelle iteration og visited-markering i BFS trænger til et "robustheds-tjek".
2. **Krone-detektion i `detect_crown.py` (Iterativ forbedring):**
   - Løsningen benytter template matching med `high_res_crown.png`. Denne metode er sårbar over for belysningsforskelle, rotationer og støj/3D-objekter. Vi skal muligvis forbedre dette med enten farvesegmentering (HSV-filtrering for gule pixels) som supplement til template matching.
3. **End-to-end Pipeline (`main.py`):**
   - Lige nu knytter `main.py` ikke modellen og scoringslogikken ordentligt sammen. Pipeline skal kalde models slut-til-slut: Billede -> Split -> Extract Features -> Classify -> Detect Crowns -> Score.

### Anbefalede Næste Skridt rækkefølge:
1. **Ret `scoring.py`:** Fikse BFS algoritmen og syntaksfejl, så vi har en pålidelig scoringsmotor (Ground Truth tests kan valideres direkte her).
2. **Forbedr og integrer `classify_tile.py` & `detect_crown.py`:** Sørg for, at klassifikationen (kNN/SVM) fungerer sammen med krone-detektoren, så vi kan generere et komplet forudsagt board.
3. **Opdater `main.py` (End-to-End Pipeline):** Bind alle modulerne sammen, så man kan give et input-billede og få en beregnet score ud i den anden ende.
4. **Model Evaluering:** Kør det på vores pre-definerede hold-out test sæt og dokumenter systemets nøjagtighed på prædikteret score ift. sand score.


## 3. Arkitektonisk Gennemgang og ML Engineerens Vurdering

Som Senior ML Engineer og co-pilot på dit speciale-projekt har jeg inspiceret hele dit repository for at sikre akademisk niveau og solid kodestruktur. Her er min vurdering af den nuværende tilstand, mangler, og den optimale rute fremad.

### Systemforståelse
Repository-strukturen er meget veludført og lever stærkt op til gode standarder for ML-forskning. Det mest fremtrædende er den korrekte håndtering af "Dataset Splitting" på spillepladeniveau (board-level hold-outs) i stedet for på tile-niveau, hvilket forhindrer data leakage betragteligt.

**Flowet ser sådan her ud:**
1. Spillepladerne indlæses og deles i $5 \times 5$ grids (`board_split.py`).
2. Hver tile får udtrukket HSV-farvefeatures (`feature_extraction.py`), der gemmes til tabulære data. 
3. kNN-modellen trænes på RGB/HSV features for multi-class terrænklassifikation (`classify_tile.py`).
4. **[Mangler]** Model opdager og tæller de gule kongekroner.
5. **[Halvfærdig]** Spillets pointsystem evaluerer hele pladen ved at gange kronerne for at opnå den endelige score pr. matchende terræn-region (`scoring.py`).

### Gaps & Fundne Mangler
1. **Modul: `detect_crown.py` (Krone-detektion)**
   - Værktøjet er kun en "stub". Detektionen af kroner mangler og vil primært afgøre spillepladens reelle points. 
   - *Anbefaling:* Standard template matching/SIFT alene giver sandsynligvis for mange FP's (False Positives) på de forskelligartede baggrunde. Vi bør bygge en robust `CrownDetector`, der inkorporerer HSV-tærskelfiltrering for gul ("gold") kombineret med en solid kontur-analyse (areal/form), så vi også kan filtrere evt. støj fra uregelmæssigt oplyste billeder uden 3D mønter.
2. **Modul: `scoring.py` (Scoring System)**
   - Algoritmen (BFS) har generelt logikken på rette sted, men implementationen forudsætter, at "crowns" er trukket fra `.csv`'en. Da `features.csv` *ikke*  indeholder kronedata, vil point pt. altid multipliceres med 0. 
   - Der mangler dynamisk kobling fra `CrownDetector`s tælling ind til selve `build_board_matrix`.
3. **Modul: `main.py` (End-to-End Inferens)**
   - Pipelinen evaluerer p.t. ikke et helt input-billede med et forward pass gennem klassifikationen *og* krone-detektionen fra ende til anden for at regne den endelige estimerede point-sum ud. 

### Trin-for-trin Plan for Færdiggørelse
For at garantere reproducerbarhed tager vi en modulær tilgang:
- **Trin 1: Implementer `CrownDetector`:** Vi færdiggør computer vision koden til krone-tælling (HSV thresholding + kontur-filtrering). Vi sikrer, at den returnerer et præcist `int` (fra 0 til 3) per tile.
- **Trin 2: Færdiggør `scoring.py`:** Vi forbedrer BFS-region detektionen, samt fjerner referencer til hardcodede CSV-labels, og overgår til objekt- / arraybaserede outputs direkte fra pipeline-logikken.
- **Trin 3: End-to-End Integration i `main.py`:** Vi lukker pipelinen med et final inferens script.
- **Trin 4: Evaluering:** Vi implementerer konkluderende test-kode og præsenterer resultaterne struktureret ift. ground truth test-sættet.